# 05-pca — Giảm chiều bằng PCA

**Câu hỏi**: PCA fit ở S0 giữ gì, bỏ gì của dữ liệu, có ổn định giữa các seed không, và lớp mới còn giữ được bao nhiêu khi chiếu vào không gian con của lớp cũ? (D10, D21, Q05.02)

### Nguồn gốc (provenance)

In commit và trạng thái dirty của code, rồi đặt `XLA_FLAGS` trước mọi lần import jax (mẫu ở `lab/README.vi.md` §11).

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit 933c9a4 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `933c9a4`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 05 và xem solver nào đã chạy

Dùng `replay.rerun` (README §5): nạp ảnh chụp sau bước 04 của `gc10_v2_ft_r0_s4`, chạy lại hàm `s05_pca`, rồi so từng khoá với `snap/05.pkl`. Quy tắc đạt: đối tượng PCA trùng mọi thuộc tính; mảng số thực lệch ≤ 1e-6 × max|giá trị|. Sau đó in solver mà PCA thực sự dùng, cùng các tham số của solver đó.

**Cách làm**: với `svd_solver="auto"`, scikit-learn 1.5.2 chọn solver theo cỡ dữ liệu (`sklearn/decomposition/_pca.py`, dòng 531–543). Gọi n là số ảnh, d là số chiều, k là số thành phần:
1. d ≤ 1000 và n ≥ 10·d → `covariance_eigh` (tất định);
2. nếu không, max(n, d) ≤ 500 → `full` (tất định);
3. nếu không, 1 ≤ k < 0.8·min(n, d) → `randomized`: chiếu lên k + 10 vector ngẫu nhiên lấy từ RNG toàn cục, rồi lặp luỹ thừa (power iteration) 7 lần nếu k < 0.1·min(n, d), 4 lần nếu không (Halko, Martinsson & Tropp 2011). Nếu n < d thì ma trận được chuyển vị trước khi chiếu.

Dự đoán từ mã nguồn, với n = 764, d = 768, k = 64: điều 1 sai (764 < 7680), điều 2 sai (768 > 500), điều 3 đúng (64 < 611.2), nên solver là `randomized`, 7 lần lặp (64 < 76.4), có chuyển vị (764 < 768).

In [2]:
# cell 1: step 05 again from the snapshot of step 04, compared with snapshot 05 (replay.rerun, QD23);
#         then the solver PCA actually used, its settings, and the auto rule of scikit-learn 1.5.2 recomputed
import numpy as np, sklearn
RUN = "gc10_v2_ft_r0_s4"
out05, ok = replay.rerun(RUN, "05")
assert ok, "step 05 does not reproduce the snapshot"
pca = out05["pca"]
n, d, k = pca.n_samples_, pca.n_features_in_, pca.n_components_
print("scikit-learn", sklearn.__version__, "| svd_solver", pca.svd_solver, "-> used", pca._fit_svd_solver,
      "| iterated_power", pca.iterated_power, "| n_oversamples", pca.n_oversamples,
      "| power_iteration_normalizer", pca.power_iteration_normalizer, "| random_state", pca.random_state)
print(f"n = {n}, d = {d}, k = {k} | rule 1 (d <= 1000 and n >= 10 d): {d <= 1000 and n >= 10 * d}"
      f" | rule 2 (max(n, d) <= 500): {max(n, d) <= 500} | rule 3 (1 <= k < 0.8 min(n, d)): {1 <= k < 0.8 * min(n, d)}")
print(f"power iterations if randomized: {7 if k < 0.1 * min(n, d) else 4} | transposed (n < d): {n < d}")
print({key: out05[key].shape for key in ("z_s0", "z_pool", "z_test0", "z_test1")})

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 05 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
  OK   pca          PCA                    max|diff| nan
  OK   z_s0         array float32          max|diff| 0
  OK   z_pool       array float32          max|diff| 0
  OK   z_test0      array float32          max|diff| 0
  OK   z_test1      array float32          max|diff| 0
rerun step 05: OK (5/5 keys)
scikit-learn 1.5.2 | svd_solver auto -> used randomized | iterated_power auto | n_oversamples 10 | power_iteration_normalizer auto | random_state None
n = 764, d = 768, k = 64 | rule 1 (d <= 1000 and n >= 10 d): False | rule 2 (max(n, d) <= 500): False | rule 3 (1 <= k < 0.8 min(n, d)): True
power iterations if randomized: 7 | transposed (n < d): True
{'z_s0': (764, 64), 'z_pool': (748, 64), 'z_test0': (241, 64), 'z_test1': (214, 64)}


- **Kết quả**:
  - `rerun` bước 05 đạt 5/5 khoá: đối tượng PCA trùng mọi thuộc tính, bốn mảng `z_*` trùng tuyệt đối (max|diff| = 0), dù driver GPU đã đổi từ 617.42 sang 621.01.
  - Solver: `svd_solver="auto"` chọn `randomized`, với `iterated_power` auto (7 lần lặp), `n_oversamples` 10, `random_state` None.
  - n = 764, d = 768, k = 64: điều 1 False, điều 2 False, điều 3 True; ma trận được chuyển vị (764 < 768).
  - Shape đầu ra: z_s0 (764, 64), z_pool (748, 64), z_test0 (241, 64), z_test1 (214, 64).
- **Ý nghĩa**:
  - Đúng như dự đoán từ mã nguồn: PCA của bước 05 là ngẫu nhiên. `random_state=None` nghĩa là PCA rút số từ RNG toàn cục của numpy, nên kết quả phụ thuộc trạng thái RNG ở lối vào bước 05. Bàn mổ đặt lại trạng thái này bằng RNG chung (QD21).
  - Vì có chuyển vị, ma trận ngẫu nhiên có một hàng cho mỗi ảnh S0. Cùng trạng thái RNG mà thứ tự ảnh S0 khác thì mỗi ảnh nhận một hàng ngẫu nhiên khác. §2.2 đo hệ quả của việc này.
  - [SUY LUẬN] D10 kết luận "PCA tất định, không phải nguồn dao động theo seed" cho CIFAR-100 (n = 20000 ≥ 10·d nên dùng `covariance_eigh`). Trên GC10, kết luận đó không còn áp dụng.

## 2. Code

### 2.1 Code của bước 05, và bước nào dùng không gian nào

In nguyên văn bằng `inspect.getsource`, không chép tay:
- `replay.s05_pca`: fit PCA trên `s0_x`, rồi chiếu pool và hai tập test;
- `MNGMMClassifier.pre_processing`: lần gọi đầu thì `fit_transform`, các lần sau thì `transform`;
- `MNGMMClassifier.test`: mọi bảng điểm đều đi qua `pre_processing`, tức là qua PCA.

Sau đó quét mã nguồn của 11 hàm bước mặc định: hàm nào đọc khoá `z_*` (không gian PCA) và hàm nào đọc `pool_x`, `test1_x` (đặc trưng thô 768 chiều).

In [3]:
# cell 2: the code of step 05 as it is (inspect.getsource), and which default steps read PCA features (z_* keys,
#         or clf.test, which applies the PCA) vs raw features (pool_x, test1_x)
import inspect, re
from classifier.mngmm import MNGMMClassifier
for f in (replay.s05_pca, MNGMMClassifier.pre_processing, MNGMMClassifier.test):
    print(f"# ---- {Path(inspect.getsourcefile(f)).name}: {f.__qualname__}\n{inspect.getsource(f)}")
for step in replay.STEPS:
    src = inspect.getsource(getattr(replay, replay.DEFAULT[step]))
    body = src.split('"""')[2] if src.count('"""') >= 2 else src      # code only, without the docstring
    in_pca = sorted(set(re.findall(r"z_\w+", body))) + (["_table -> clf.test"] if "_table(" in body else [])
    in_raw = sorted(set(re.findall(r'"(pool_x|test1_x|s0_x|test0_x|test_all_x|test_old_x)"', body)))
    print(f"{step} {replay.DEFAULT[step]:16s} PCA space: {in_pca or '-'} | raw features: {in_raw or '-'}")

# ---- replay.py: s05_pca
def s05_pca(st):
    """05 PCA to num_dim, fit on the labeled S0 data (mngmm.pre_processing: fit_transform on the first call; the
    randomized solver draws the global RNG). In: s0_x, s0_y, pool_x, test0_x, test1_x.
    Out: pca, z_s0, z_pool, z_test0, z_test1 (num_dim features)."""
    clf = _clf(st)
    z_s0, _ = clf.pre_processing(st["s0_x"], st["s0_y"])
    t = clf.pca.transform
    return dict(pca=clf.pca, z_s0=z_s0, z_pool=t(st["pool_x"]), z_test0=t(st["test0_x"]), z_test1=t(st["test1_x"]))

# ---- mngmm.py: MNGMMClassifier.pre_processing
    def pre_processing(self, features, labels):
        if self.pca is None:
            self.pca = PCA(n_components=self.num_dim)
            features = self.pca.fit_transform(features)
        else:
            features = self.pca.transform(features)
        return features, labels

# ---- mngmm.py: MNGMMClassifier.test
    def test(self, test_features, test_labels):

        test_features, test_labels = self.pre_pro

- **Kết quả**:
  - `s05_pca` gọi `pre_processing` một lần trên `s0_x` (764 ảnh S0). Lần gọi này `fit_transform` vì `clf.pca` đang là None. Sau đó `pca.transform` chiếu `pool_x`, `test0_x`, `test1_x`.
  - `pre_processing` tạo `PCA(n_components=self.num_dim)` mà không truyền `svd_solver` hay `random_state`, nên dùng mặc định `auto` và `None` (khớp §1.1).
  - `test` luôn gọi `pre_processing` trước, tức là mọi bảng điểm đều chấm trong không gian PCA.
  - Quét 11 bước:
    - đọc `z_*`: 06 (`z_s0`, `z_test0`), 11 và 13 (`z_pool`, `z_test1`), 12 (`z_pool`);
    - đi qua `clf.test`: 08 và 14, đọc tập test thô rồi để `test` chiếu qua PCA;
    - đọc đặc trưng thô, không qua PCA: 09 (`pool_x`) và 10 (`test1_x`);
    - 04 và 07 không đọc đặc trưng.
- **Ý nghĩa**:
  - Pipeline chạy trên hai không gian. Phân cụm (09) và đặt tên cụm (10) dùng đặc trưng thô 768 chiều. Mọi Gaussian (06, 11, 13), bộ lọc (12) và chấm điểm (08, 14) dùng không gian PCA 64 chiều fit ở 5 lớp cũ.
  - Vì vậy PCA không ảnh hưởng tới nhãn giả của bước 09. Nó chỉ ảnh hưởng tới việc học và dùng nhãn giả ở các bước 11–14.
  - Paper (Appendix C.3) chỉ nói "PCA 384 chiều" và "k-means", không nói k-means chạy trên không gian nào. Chạy k-means trên đặc trưng thô là lựa chọn của code. Ghi lại để so với paper (paper > code).

### 2.2 PCA ngẫu nhiên xê dịch không gian con bao nhiêu

So ba lần fit trên cùng 764 ảnh S0 của ảnh chụp:
- (a) solver `randomized`, trạng thái RNG ở lối vào bước 05 của ảnh chụp, thứ tự hàng gốc. Phải trùng PCA của ảnh chụp; đây là phép kiểm cho cách làm;
- (b) như (a), nhưng các hàng S0 theo một hoán vị cố định (`RandomState(0)`, không đụng RNG toàn cục);
- (c) solver `full` (SVD đầy đủ, tất định): đây là không gian con 64 chiều tối ưu.

**Cách làm**: so hai không gian con bằng góc chính (principal angles; Björck & Golub 1973). Với U_a, U_b là hai ma trận 768 × 64 có cột trực chuẩn, góc chính thứ i là θ_i = arccos σ_i(U_aᵀ U_b), trong đó σ_i là giá trị kỳ dị thứ i. θ = 0° nghĩa là trùng hướng, θ = 90° nghĩa là vuông góc. Kèm theo, |cos| giữa từng cặp thành phần cùng thứ tự, và phương sai S0 mà mỗi không gian con giữ được.

Cùng lúc in các trị riêng λ_k của S0 quanh k = 64. Theo định lý Davis–Kahan (Davis & Kahan 1970), khoảng cách giữa λ_64 và λ_65 càng nhỏ so với nhiễu thì không gian con 64 chiều càng dễ xê dịch. Vì vậy các thành phần cuối, nơi các trị riêng sát nhau, được dự đoán là kém ổn định nhất.

In [4]:
# cell 3: how far the random PCA of step 05 moves the 64-d subspace: (a) randomized, snapshot RNG state, original row order
#         (must equal the snapshot), (b) same RNG state, S0 rows permuted, (c) the exact full SVD; principal angles
from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles
snap_pca = out05["pca"]                                            # the PCA of snapshot 05 (cell 1)
ncomp = int(snap_pca.n_components_)                                # 64 (num_dim of GC10)                                  
blobs = replay.snapshots(RUN)
s0 = np.asarray(blobs["04"]["out"]["s0_x"])                       # (764, 768), row order of the snapshot
def fit(A, solver="auto"):
    np.random.set_state(blobs["05"]["rng_in"])                    # common random numbers: RNG state at the entry of 05
    return PCA(n_components=ncomp, svd_solver=solver).fit(A)
pa = fit(s0)
pb = fit(s0[np.random.RandomState(0).permutation(len(s0))])
pc = fit(s0, "full")
print("(a) = snapshot PCA:", np.array_equal(pa.components_, snap_pca.components_))
tot = pc.explained_variance_[0] / pc.explained_variance_ratio_[0]     # total variance of S0
for name, P in (("(a) vs (b): same RNG, rows permuted", pb), ("(a) vs (c): randomized vs full", pc)):
    ang = np.degrees(subspace_angles(pa.components_.T, P.components_.T))
    cos = np.abs((pa.components_ * P.components_).sum(1))
    print(f"{name}: largest angle {ang.max():.2f} deg, median {np.median(ang):.3f} deg, angles > 10 deg: {(ang > 10).sum()}/{ncomp}"
          f" | |cos| of component 1, 10, 32, 64: {cos[0]:.4f} {cos[9]:.4f} {cos[31]:.4f} {cos[63]:.4f}")
print("S0 variance kept: (a) {:.4f}  (b) {:.4f}  (c) {:.4f}  of total {:.2f}".format(
      *(P.explained_variance_.sum() / tot for P in (pa, pb, pc)), tot))
lam = PCA(n_components=80, svd_solver="full").fit(s0).explained_variance_
print("eigenvalues 60-68 of S0:", np.round(lam[59:68], 4), "| relative gap (l64 - l65) / l64:", round((lam[63] - lam[64]) / lam[63], 4))

(a) = snapshot PCA: True
(a) vs (b): same RNG, rows permuted: largest angle 41.53 deg, median 0.003 deg, angles > 10 deg: 4/64 | |cos| of component 1, 10, 32, 64: 1.0000 1.0000 1.0000 0.3803
(a) vs (c): randomized vs full: largest angle 17.43 deg, median 0.001 deg, angles > 10 deg: 2/64 | |cos| of component 1, 10, 32, 64: 1.0000 1.0000 1.0000 0.9486
S0 variance kept: (a) 0.8621  (b) 0.8620  (c) 0.8622  of total 242.57
eigenvalues 60-68 of S0: [0.4547 0.4286 0.4253 0.4175 0.4117 0.4057 0.4024 0.3955 0.3867] | relative gap (l64 - l65) / l64: 0.0147


- **Kết quả**:
  - (a) trùng PCA của ảnh chụp (True): cách làm tái lập đúng bước 05.
  - (a) so với (b), cùng trạng thái RNG nhưng hoán vị hàng S0:
    - góc chính lớn nhất 41.53°, trung vị 0.003°, 4/64 góc lớn hơn 10°;
    - |cos| của thành phần 1, 10, 32 là 1.0000; của thành phần 64 là 0.3803.
  - (a) so với (c), randomized so với full: góc lớn nhất 17.43°, trung vị 0.001°, 2/64 góc lớn hơn 10°; |cos| của thành phần 64 là 0.9486.
  - Phần phương sai S0 được giữ: (a) 0.8621, (b) 0.8620, (c) 0.8622, trên tổng 242.57.
  - Trị riêng 60–68 của S0: 0.4547, 0.4286, 0.4253, 0.4175, 0.4117, 0.4057, 0.4024, 0.3955, 0.3867. Khoảng cách tương đối (λ64 − λ65)/λ64 = 0.0147.
- **Ý nghĩa**:
  - Không gian con 64 chiều gần như cố định. Hơn 60 hướng trùng nhau tới vài phần nghìn độ. Chỉ vài hướng ở cuối phổ xoay mạnh, lớn nhất 41.5° khi đổi thứ tự hàng.
  - Khớp Davis–Kahan: quanh k = 64 các trị riêng sát nhau, λ64 và λ65 chỉ cách 1.5%. Vì vậy tách hướng thứ 64 khỏi hướng thứ 65 không bền, còn các hướng đầu có khoảng cách lớn nên không xoay.
  - Theo phương sai, ảnh hưởng rất nhỏ: các lần fit giữ 0.8620–0.8622 tổng phương sai. Mỗi hướng cuối mang khoảng 0.41/242.57 ≈ 0.17% phương sai (tự chia).
  - Như vậy D10 trên GC10 cần sửa thành: PCA phụ thuộc seed, kể cả qua thứ tự hàng, nhưng chỉ ở vài hướng cuối mang rất ít phương sai.
  - [SUY LUẬN] Ảnh hưởng xuống các bước sau chưa đo được ở đây. Gaussian học được gần Σ = I (D23), nên mỗi hướng gần như đóng góp ngang nhau vào khoảng cách. Vài hướng xoay chỉ đổi một phần nhỏ của khoảng cách đó, khớp với việc S0_All giữa các seed chỉ lệch tới 3/241 ảnh (r1, r2). §5.1 đo trên 15 ảnh chụp.
  - Trị riêng ở cuối không gian con chỉ khoảng 0.4 < 1, nên các hướng cuối có phương sai nhỏ hơn 1, là điểm khởi đầu của §3.2.

## 3. Phương sai

### 3.1 PCA của S0 giữ bao nhiêu phương sai của mỗi tập

Không gian con được fit trên 764 ảnh của 5 lớp cũ, rồi dùng để chiếu cả ảnh lớp mới. Câu hỏi: không gian đó giữ được bao nhiêu phương sai của từng tập (S0, pool, test lớp cũ, test lớp mới), và lượng đó đổi thế nào theo số thành phần.

**Cách làm**: với tập X (n × 768) và P là ma trận chiếu lên m thành phần đầu (m = 8, 16, 32, 64), phần giữ lại là ‖(X − x̄)Pᵀ‖²_F / ‖X − x̄‖²_F. Ở đây x̄ là trung bình của chính tập X, không phải trung bình của S0, nên đại lượng này đo hình dạng của tập, không đo khoảng cách từ tập tới S0 (QD24, để Q12.02 còn sạch). Với S0, kết quả ở m = 64 phải trùng tổng `explained_variance_ratio_` (0.8621, §2.2), dùng làm phép kiểm. Dùng PCA của ảnh chụp (`snap_pca`), không fit lại.

In [5]:
# cell 4: share of each set's variance (about its own mean) kept by the first m components of the snapshot PCA
#         (fit on S0 only): S0, pool, test0 (old classes), test1 (novel classes); m = 8, 16, 32, 64
import pandas as pd
out04 = blobs["04"]["out"]
sets31 = {"S0": out04["s0_x"], "pool": out04["pool_x"], "test0 (old)": out04["test0_x"], "test1 (novel)": out04["test1_x"]}
comp31 = snap_pca.components_                                      # (64, 768), rows orthonormal
rows31 = {}
for name31, X in sets31.items():
    Xc = np.asarray(X, dtype=np.float64) - np.asarray(X, dtype=np.float64).mean(0)
    total = (Xc ** 2).sum()
    rows31[name31] = {f"m={m}": ((Xc @ comp31[:m].T) ** 2).sum() / total for m in (8, 16, 32, 64)}
    rows31[name31]["total var"] = total / (len(Xc) - 1)
print(pd.DataFrame(rows31).T.round(4).to_string())
print("check: S0 at m=64 vs explained_variance_ratio_ sum:", round(rows31["S0"]["m=64"], 4), round(snap_pca.explained_variance_ratio_.sum(), 4))

                  m=8    m=16    m=32    m=64  total var
S0             0.5864  0.6847  0.7791  0.8621   242.5674
pool           0.3609  0.4661  0.5713  0.6847   167.3955
test0 (old)    0.5620  0.6488  0.7307  0.8019   239.2227
test1 (novel)  0.3243  0.4261  0.5387  0.6631   157.7288
check: S0 at m=64 vs explained_variance_ratio_ sum: 0.8621 0.8621


- **Kết quả** (phần phương sai giữ lại, quanh trung bình của chính tập đó):

  | tập | m = 8 | m = 16 | m = 32 | m = 64 | tổng phương sai |
  |---|---|---|---|---|---|
  | S0 | 0.586 | 0.685 | 0.779 | 0.862 | 242.6 |
  | pool | 0.361 | 0.466 | 0.571 | 0.685 | 167.4 |
  | test0 (lớp cũ) | 0.562 | 0.649 | 0.731 | 0.802 | 239.2 |
  | test1 (lớp mới) | 0.324 | 0.426 | 0.539 | 0.663 | 157.7 |

  Phép kiểm: S0 ở m = 64 là 0.8621, trùng tổng `explained_variance_ratio_`.
- **Ý nghĩa**:
  - Không gian con của S0 giữ 86% phương sai của S0 và 80% của test lớp cũ, nhưng chỉ giữ 66–68% của tập lớp mới (test1) và của pool (gồm 698/748 ảnh lớp mới). Khoảng một phần ba phương sai của lớp mới nằm ngoài 64 hướng của lớp cũ. Đây là bằng chứng bằng số cho điều D10 lo ("lớp mới có thể mất phương sai, nhất là khi có ít lớp cơ sở").
  - Khoảng cách lớn nhất ở m nhỏ: m = 8 giữ 59% của S0 nhưng chỉ 32% của test1. Các hướng đầu là hướng của lớp cũ; lớp mới trải ra theo những hướng khác.
  - Lớp cũ chênh S0 với test0 (0.862 so với 0.802) là mức giảm thường gặp khi chuyển từ dữ liệu fit sang dữ liệu mới cùng phân phối. Lớp mới chênh 0.862 so với 0.663, tức gấp khoảng ba lần mức đó.
  - [SUY LUẬN] Phương sai mất đi chưa chắc là thông tin phân biệt mất đi, vì phần bị bỏ có thể là phương sai trong lớp. E03.02 cho thấy cả hai chiều: probe trên PCA-64 thua probe thô ở r1, r2 nhưng thắng ở r0. §4.1 tách phần này: phương sai trong lớp và khoảng cách giữa các tâm.
  - Phân cụm (09) chạy trên đặc trưng thô nên không chịu mất mát này (§2.1). Nó chỉ chạm các Gaussian và bộ lọc ở 11–14.

### 3.2 Thang đo của từng lớp cũ trong không gian PCA

Bước 06 học một Gaussian N(μ_c, Σ_c) cho mỗi lớp cũ, khởi tạo Σ_c = I, và luật dừng của 07 so det Σ với ngưỡng (D14). Theo D23, Σ học được chỉ rời I một khoảng nhỏ về phía hiệp phương sai của dữ liệu. Vậy det Σ học được lớn hơn hay nhỏ hơn 1 phụ thuộc vào thang đo của dữ liệu: phương sai trong lớp của mỗi hướng lớn hơn hay nhỏ hơn 1.

**Cách làm**: với mỗi lớp cũ c, tính hiệp phương sai mẫu (sample covariance) S_c của `z_s0` thuộc lớp c (64 × 64). In ra:
- số hướng có phương sai trong lớp lớn hơn 1 (đường chéo của S_c);
- tr(S_c)/64, phương sai trung bình mỗi hướng;
- log det S_c = Σ log λ_i, với λ_i là các trị riêng của S_c.

Thêm hàng "gộp": hiệp phương sai trong lớp gộp (pooled within-class) của 5 lớp. Nhãn S0 là nhãn thật mà mô hình được dùng, nên không có vấn đề nhìn trước.

Dự đoán [SUY LUẬN, từ trị riêng ở §2.2: tổng phương sai 0.4 ở hướng thứ 64]: chỉ vài hướng đầu có phương sai trong lớp lớn hơn 1; log det S_c âm rõ, nên Σ học ở 06 có det < 1, tức nhánh "det < 1" của luật dừng (notebook cũ đo được det trung bình của lớp cũ là 0.3707). Kiểm ở 06 và 07.

In [6]:
# cell 5: scale of each old class in the PCA space (z_s0, S0 labels): directions with within-class variance > 1,
#         mean variance per direction, log det of the sample covariance; plus the pooled within-class covariance
import json
names32 = json.loads(Path("splits/gc10_split.json").read_text())["classes"]
z32, y32 = np.asarray(out05["z_s0"], dtype=np.float64), np.asarray(out04["s0_y"]).astype(int)
rows32, Sw = {}, np.zeros((ncomp, ncomp))
for c in np.unique(y32):
    Zc = z32[y32 == c]
    S = np.cov(Zc.T)                                               # (64, 64), divisor n_c - 1
    Sw += S * (len(Zc) - 1)
    rows32[names32[c]] = {"n": len(Zc), "dirs var > 1": int((np.diag(S) > 1).sum()),
                          "tr(S)/64": np.trace(S) / ncomp, "log det S": np.linalg.slogdet(S)[1]}
Sw /= len(z32) - len(np.unique(y32))
rows32["pooled"] = {"n": len(z32), "dirs var > 1": int((np.diag(Sw) > 1).sum()),
                    "tr(S)/64": np.trace(Sw) / ncomp, "log det S": np.linalg.slogdet(Sw)[1]}
print(pd.DataFrame(rows32).T.round(3).to_string())
print("pooled within-class variance of components 1-8:", np.round(np.diag(Sw)[:8], 3))

                   n  dirs var > 1  tr(S)/64  log det S
punching_hole  137.0          24.0     1.426    -80.179
welding_line   170.0          32.0     2.146    -35.021
crescent_gap   143.0          29.0     1.489    -66.070
water_spot     184.0          32.0     1.872    -48.360
oil_spot       130.0          32.0     1.976    -76.408
pooled         764.0          32.0     1.799      9.585
pooled within-class variance of components 1-8: [6.848 8.061 5.44  6.163 6.467 6.402 4.705 4.414]


- **Kết quả**:

  | lớp | n | số hướng có phương sai > 1 | tr(S)/64 | log det S |
  |---|---|---|---|---|
  | punching_hole | 137 | 24 | 1.43 | −80.2 |
  | welding_line | 170 | 32 | 2.15 | −35.0 |
  | crescent_gap | 143 | 29 | 1.49 | −66.1 |
  | water_spot | 184 | 32 | 1.87 | −48.4 |
  | oil_spot | 130 | 32 | 1.98 | −76.4 |
  | gộp | 764 | 32 | 1.80 | +9.6 |

  Phương sai trong lớp gộp của thành phần 1–8: 6.85, 8.06, 5.44, 6.16, 6.47, 6.40, 4.71, 4.41.
- **Ý nghĩa**:
  - Dự đoán sai một nửa. Không phải "chỉ vài hướng" có phương sai trong lớp lớn hơn 1: có 24–32/64 hướng, và phương sai trung bình mỗi hướng là 1.4–2.1, đều lớn hơn 1. Phần đúng: log det S_c của từng lớp âm rõ (−35 tới −80).
  - Hai điều đó cùng đúng được vì log det là tổng log của các trị riêng, nên vài trị riêng rất nhỏ kéo cả tổng xuống. Mỗi lớp chỉ có 130–184 ảnh cho 64 chiều. Theo luật Marchenko–Pastur (Marchenko & Pastur 1967), khi tỉ số d/n lớn, các trị riêng mẫu nhỏ nhất bị kéo về gần 0. Ví dụ oil_spot: d/n = 64/130 ≈ 0.49, nên trị riêng nhỏ nhất co còn khoảng (1 − √0.49)² ≈ 0.09 lần giá trị thật. Khi gộp cả 764 ảnh, d/n ≈ 0.08, phép co yếu đi và log det thành +9.6. Đây là phía "n gần d" của D21 (sụp hoàn toàn khi n ≤ d).
  - Hệ quả cho luật dừng: thang đo của dữ liệu không quyết định được một mình det Σ học ở 06 lớn hay nhỏ hơn 1. Một nửa số hướng muốn tăng, một nửa muốn giảm. Vậy mà notebook cũ đo det trung bình học được của lớp cũ là 0.3707 < 1, tức có cơ chế khác kéo det xuống.
  - [SUY LUẬN, cần kiểm ở 06] Một ứng viên: Adam đẩy mọi phần tử của Σ, kể cả 64 × 63 phần tử ngoài đường chéo, mỗi bước một lượng cỡ learning rate (D23). Với Σ = I + E nhỏ, log det(I + E) ≈ tr E − ½‖E‖²_F. Phần ngoài đường chéo không vào tr E nhưng vào ‖E‖²_F, nên luôn kéo log det xuống. Ví dụ: nếu 4032 phần tử ngoài đường chéo đều cỡ 0.03 (cỡ D23 đo được) thì ½‖E‖²_F ≈ ½ × 4032 × 0.0009 ≈ 1.8, trong khi tr E tối đa khoảng 64 × 0.033 ≈ 2.1. Hai số cùng cỡ, nên dấu của log det phụ thuộc phần ngoài đường chéo. Ở 06 sẽ tách log det của `params0` thành tr E và ‖E‖²_F.
  - Suy luận cũ của D14 ("nhánh của luật dừng đi theo thang đo đặc trưng") vì vậy chưa đủ. Ghi lại để kiểm ở 06 và 07.

## 4. Lớp mới trong không gian con của S0

### 4.1 Phương sai trong lớp và khoảng cách giữa các tâm còn giữ được

§3.1 cho thấy không gian con của S0 giữ 66–68% phương sai của tập lớp mới. Phần mất đi gồm hai loại: phương sai trong lớp (không cần để phân biệt lớp) và khoảng cách giữa các lớp (cần để phân biệt). Tách hai phần này theo từng lớp mới.

**Cách làm** (nhãn thật của pool chỉ dùng để chấm; QD24):
- phương sai trong lớp giữ lại của lớp c: ‖(X_c − x̄_c)Pᵀ‖²_F / ‖X_c − x̄_c‖²_F, với X_c là ảnh pool của lớp mới c và P là 64 thành phần của `snap_pca`;
- khoảng cách giữa tâm lớp a và tâm lớp b giữ lại: ‖P(x̄_a − x̄_b)‖ / ‖x̄_a − x̄_b‖, cho 10 cặp lớp mới;
- làm tương tự cho 10 cặp lớp cũ (tâm tính trên S0) để so sánh.

Không tính phần dư của từng ảnh, cũng không tính chỉ số tách cũ/mới nào (QD24: để Q12.02 còn sạch).

In [7]:
# cell 6: novel classes inside the S0 subspace (true pool labels: scoring only): within-class variance kept per novel class,
#         and the share of each between-centre distance kept (novel pairs on the pool; old pairs on S0, for reference)
from itertools import combinations
P41 = snap_pca.components_.astype(np.float64)                      # (64, 768)
y41 = np.asarray(out04["true_pool_y"]).astype(int)
X41 = np.asarray(out04["pool_x"], dtype=np.float64)
S041 = np.asarray(out04["s0_x"], dtype=np.float64)
within41, centre = {}, {}
for c in range(5, 10):
    Xc = X41[y41 == c]
    D = Xc - Xc.mean(0)
    within41[names32[c]] = {"n pool": len(Xc), "within var kept": ((D @ P41.T) ** 2).sum() / (D ** 2).sum()}
    centre[c] = Xc.mean(0)
for c in range(5):
    centre[c] = S041[y32 == c].mean(0)
print(pd.DataFrame(within41).T.round(3).to_string())
kept = lambda a, b: np.linalg.norm(P41 @ (centre[a] - centre[b])) / np.linalg.norm(centre[a] - centre[b])
for label, ids in (("novel pairs (pool)", range(5, 10)), ("old pairs (S0)", range(5))):
    pairs = {f"{names32[a][:6]}-{names32[b][:6]}": round(kept(a, b), 3) for a, b in combinations(ids, 2)}
    print(f"{label}: distance kept {pairs} | min {min(pairs.values())}, median {np.median(list(pairs.values())):.3f}")

               n pool  within var kept
silk_spot       420.0            0.605
inclusion       135.0            0.685
rolled_pit       17.0            0.560
crease           32.0            0.537
waist_folding    94.0            0.534
novel pairs (pool): distance kept {'silk_s-inclus': np.float64(0.9), 'silk_s-rolled': np.float64(0.942), 'silk_s-crease': np.float64(0.926), 'silk_s-waist_': np.float64(0.935), 'inclus-rolled': np.float64(0.933), 'inclus-crease': np.float64(0.928), 'inclus-waist_': np.float64(0.924), 'rolled-crease': np.float64(0.856), 'rolled-waist_': np.float64(0.842), 'crease-waist_': np.float64(0.857)} | min 0.842, median 0.925
old pairs (S0): distance kept {'punchi-weldin': np.float64(0.999), 'punchi-cresce': np.float64(1.0), 'punchi-water_': np.float64(1.0), 'punchi-oil_sp': np.float64(1.0), 'weldin-cresce': np.float64(1.0), 'weldin-water_': np.float64(1.0), 'weldin-oil_sp': np.float64(1.0), 'cresce-water_': np.float64(1.0), 'cresce-oil_sp': np.float64(1.0), 'water_-

- **Kết quả**:
  - Phần phương sai trong lớp giữ lại (ảnh pool): silk_spot 0.605 (420 ảnh), inclusion 0.685 (135), rolled_pit 0.560 (17), crease 0.537 (32), waist_folding 0.534 (94).
  - Phần khoảng cách giữa các tâm lớp mới giữ lại: 0.842–0.942, trung vị 0.925. Ba cặp thấp nhất là rolled–waist 0.842, rolled–crease 0.856, crease–waist 0.857. Các cặp có silk hoặc inclusion giữ 0.900–0.942.
  - Cặp lớp cũ (tâm trên S0): 0.999–1.000.
- **Ý nghĩa**:
  - Với lớp mới, không gian con của S0 bỏ đi nhiều phương sai trong lớp (giữ 53–69%) hơn khoảng cách giữa các tâm (giữ 84–94%). Phần mất ở §3.1 chủ yếu là độ trải trong lớp, không phải khoảng cách giữa lớp.
  - Ba cặp mất khoảng cách nhiều nhất là crease, rolled_pit, waist_folding. Đây đúng là bộ ba mà k-means gộp thành một cụm ở r0 s4 (27/32 crease và 16/17 rolled nằm trong cụm của waist; notebook cũ, cell 4). Không gian con của lớp cũ ít "thấy" hướng phân biệt ba lớp này nhất.
  - [SUY LUẬN] Theo khoảng cách Euclid, tỉ lệ khoảng cách trên độ trải còn tăng sau khi chiếu. Ví dụ crease–waist: khoảng cách giữ 0.857, độ lệch chuẩn trong lớp giữ khoảng √0.536 ≈ 0.73, nên tỉ lệ tăng khoảng 0.857/0.73 ≈ 1.17 lần. Như vậy PCA-64 chưa chắc làm ba lớp này khó tách hơn, khớp với E03.02 ở r0 (probe trên PCA-64 0.641, probe thô 0.611). Đây là phép đo Euclid trên tâm và tổng phương sai, chưa tính hướng của hiệp phương sai, nên chưa đủ để kết luận về khả năng tách.
  - Cặp lớp cũ giữ gần 1.000 là điều hiển nhiên: PCA fit trên chính các ảnh này, và hướng giữa các tâm lớp mang phần lớn phương sai, nên nằm trong 64 hướng đầu. Con số này chỉ là mốc, không phải kết quả.
  - Phân cụm (09) chạy trên đặc trưng thô nên không chịu mất mát này. Mất mát tác động ở 11–14, nơi Gaussian và bộ lọc phân biệt ba lớp trên trong không gian 64 chiều.

## 5. Qua 15 ảnh chụp

### 5.1 PCA khác nhau bao nhiêu giữa các seed

Trong một FT seed, 5 seed phase 2 dùng cùng 764 ảnh S0 với cùng đặc trưng (`04-loader` §3.2). Chỉ thứ tự hàng và trạng thái RNG ở lối vào bước 05 là khác. §2.2 cho thấy một lần hoán vị hàng đã xoay vài hướng cuối tới 41.5°. Ở đây đo trên cả 15 ảnh chụp.

**Cách làm**: đọc `snap/04.pkl` và `snap/05.pkl` của 15 run bằng pickle. Trong mỗi FT seed:
- góc chính giữa PCA của 10 cặp seed;
- góc chính giữa PCA của mỗi seed và PCA `full` (tối ưu, tất định) trên cùng S0;
- phần phương sai S0 được giữ.

Không so giữa các FT seed, vì đặc trưng của mỗi r là một không gian khác.

In [8]:
# cell 7: PCA of step 05 across the 15 snapshots (pickle): within each FT seed, principal angles between the 5 seeds' PCA
#         (10 pairs) and between each seed's PCA and the exact full-SVD PCA of the same S0; S0 variance kept
import pickle
snap51 = {}
for d in sorted(Path("runs").glob("*__gc10_v2_ft_r[0-2]_s[0-4]")):
    tag51 = d.name.split("__")[1][len("gc10_v2_ft_"):]               # r0_s0 ... r2_s4
    snap51[tag51] = (pickle.loads((d / "snap" / "04.pkl").read_bytes())["out"]["s0_x"],
                     pickle.loads((d / "snap" / "05.pkl").read_bytes())["out"]["pca"])
assert len(snap51) == 15, len(snap51)
deg51 = lambda A, B: np.degrees(subspace_angles(A.components_.T, B.components_.T))
for r in range(3):
    tags = [f"r{r}_s{s}" for s in range(5)]
    full51 = PCA(n_components=ncomp, svd_solver="full").fit(snap51[tags[0]][0])
    pair_max = [deg51(snap51[a][1], snap51[b][1]).max() for a, b in combinations(tags, 2)]
    pair_n10 = [(deg51(snap51[a][1], snap51[b][1]) > 10).sum() for a, b in combinations(tags, 2)]
    to_full = [deg51(snap51[t][1], full51) for t in tags]
    kept51 = [snap51[t][1].explained_variance_ratio_.sum() for t in tags]
    print(f"r{r}: between seeds (10 pairs) largest angle {min(pair_max):.1f}-{max(pair_max):.1f} deg, angles > 10 deg per pair "
          f"{min(pair_n10)}-{max(pair_n10)} | vs full: largest angle per seed {[round(a.max(), 1) for a in to_full]}, "
          f"median {max(np.median(a) for a in to_full):.4f} deg at most | S0 variance kept {min(kept51):.4f}-{max(kept51):.4f}")

r0: between seeds (10 pairs) largest angle 21.7-39.1 deg, angles > 10 deg per pair 3-4 | vs full: largest angle per seed [np.float32(19.4), np.float32(26.1), np.float32(18.3), np.float32(34.3), np.float32(17.4)], median 0.0010 deg at most | S0 variance kept 0.8621-0.8622
r1: between seeds (10 pairs) largest angle 28.0-79.3 deg, angles > 10 deg per pair 3-4 | vs full: largest angle per seed [np.float32(23.0), np.float32(56.8), np.float32(44.5), np.float32(38.9), np.float32(32.7)], median 0.0013 deg at most | S0 variance kept 0.8470-0.8471
r2: between seeds (10 pairs) largest angle 21.9-45.0 deg, angles > 10 deg per pair 3-4 | vs full: largest angle per seed [np.float32(24.3), np.float32(31.4), np.float32(23.7), np.float32(19.7), np.float32(14.8)], median 0.0021 deg at most | S0 variance kept 0.8447-0.8448


- **Kết quả** (trong mỗi FT seed, 5 seed phase 2):

  | FT seed | góc lớn nhất giữa 2 seed (10 cặp) | số góc > 10° mỗi cặp | góc lớn nhất so với `full`, theo seed 0–4 | trung vị góc so với `full` | phương sai S0 giữ lại |
  |---|---|---|---|---|---|
  | r0 | 21.7–39.1° | 3–4 | 19.4, 26.1, 18.3, 34.3, 17.4° | ≤ 0.0010° | 0.8621–0.8622 |
  | r1 | 28.0–79.3° | 3–4 | 23.0, 56.8, 44.5, 38.9, 32.7° | ≤ 0.0013° | 0.8470–0.8471 |
  | r2 | 21.9–45.0° | 3–4 | 24.3, 31.4, 23.7, 19.7, 14.8° | ≤ 0.0021° | 0.8447–0.8448 |
- **Ý nghĩa**:
  - Điều thấy ở §2.2 với một hoán vị lặp lại ở cả 15 ảnh chụp: giữa hai seed bất kỳ có 3–4 hướng xoay hơn 10°, còn các hướng khác trùng tới vài phần nghìn độ. Phần phương sai giữ lại bằng nhau tới 4 chữ số thập phân.
  - Độ lớn của góc khác nhau theo FT seed: r1 có một hướng xoay tới 79.3°, gần vuông góc. [SUY LUẬN] Có thể khoảng cách giữa các trị riêng quanh k = 64 của r1 còn nhỏ hơn r0 (r0: 1.5%, §2.2); chưa đo cho r1.
  - So với PCA tối ưu (`full`), mỗi seed lệch 15–57° ở hướng xa nhất. Như vậy không seed nào cho đúng không gian con tối ưu, và độ lệch không giống nhau giữa các seed.
  - Kết luận cho D10 trên GC10: bước 05 phụ thuộc seed, qua trạng thái RNG và qua thứ tự hàng S0, nhưng chỉ ở 3–4 hướng cuối, mang rất ít phương sai (§2.2). Ảnh hưởng xuống các bước sau đo ở 06 và 08.
  - Phần phương sai giữ lại khác nhau giữa các FT seed (0.862 ở r0, 0.847 ở r1, 0.845 ở r2). Đó là khác biệt của đặc trưng, không phải của PCA.

## Summary (EN)

1. **Step 05 reproduces exactly and is randomized.** Rerunning step 05 of `gc10_v2_ft_r0_s4` gives the PCA object and all four projected sets bit-identical to the snapshot. With n = 764, d = 768, k = 64, scikit-learn 1.5.2's `auto` rule picks the `randomized` solver (7 power iterations, transposed input, `random_state=None`), so the PCA draws the global RNG (§1.1). Clustering (09) and cluster naming (10) read the raw 768-d features; all Gaussians, the filter and the scoring (06, 08, 11–14) work in the 64-d PCA space (§2.1).
2. **The random PCA moves only a few tail directions.** With the snapshot's RNG state, permuting the S0 rows rotates 4 of 64 principal angles by more than 10° (largest 41.5°, median 0.003°), while the kept S0 variance stays 0.862. Eigenvalues near k = 64 are nearly tied (relative gap λ64 → λ65 of 1.5%), as Davis–Kahan predicts for unstable directions (§2.2). Across the 15 snapshots, every pair of seeds within an FT seed differs by 3–4 such directions (largest 21.7–79.3°) and no seed reaches the exact subspace (§5.1). [inference] D10's "PCA is deterministic" holds for CIFAR-100, not for GC10.
3. **Novel classes lose a third of their variance, mostly spread.** The S0 subspace keeps 0.862 of S0 variance and 0.802 of the old test set, but 0.663 of the novel test set and 0.685 of the pool (§3.1). Per novel class it keeps 0.53–0.69 of the within-class variance but 0.84–0.94 of the between-centre distances. The pairs that lose most distance are crease, rolled_pit and waist_folding (0.842–0.857), the trio k-means merges (§4.1). [inference] In Euclidean terms projection raises distance relative to spread, so PCA-64 need not make them harder to separate.
4. **Per-class scale does not fix the early-stop branch.** In PCA space each old class has 24–32 of 64 directions with within-class variance above 1 (mean variance 1.4–2.1), yet log det of its sample covariance is −35 to −80, because 130–184 images in 64 dimensions shrink the smallest eigenvalues (Marchenko–Pastur); pooled over 764 images it is +9.6 (§3.2). The prediction "few directions above 1" was wrong. [inference] Why the learned covariances end with det < 1 (0.3707 in the old notebook) needs the off-diagonal drift of SVI, checked in 06.